# LLM-like architecture with pyTorch

Most code from chapter 4 from [thelmbook.com](https://www.thelmbook.com/). Sources:
- ...

In [ ]:
from tqdm import tqdm   # For progress bars

import torch
import torch.nn as nn
from transformers import AutoTokenizer

import utils
import thelmbook as tlmb

In [ ]:
import yaml

with open("params.yaml") as f:
    params = yaml.safe_load(f)['llm']

# define explicitly to reduce chenges to original code
data_url = params['data_url']
emb_dim = params['emb_dim']
num_heads = params['num_heads']
num_blocks = params['num_blocks']
batch_size = params['batch_size']
learning_rate = params['learning_rate']
num_epochs = params['num_epochs']
context_size = params['context_size']

utils.print_params(params)

In [ ]:
tlmb.set_seed(params['random_state'])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Initialize the tokenizer using Microsoft's Phi-3.5-mini model
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct")
# Get padding token index for padding shorter sequences
pad_idx = tokenizer.pad_token_id
pad_idx

In [ ]:
# Set evaluation interval (number of examples after which to perform validation)
# 200,000 examples provides a good balance between training time and monitoring frequency
eval_interval = 200_000
examples_processed = 0  # Counter for tracking progress toward next evaluation

# Define test contexts for generating sample text during evaluation
contexts = [
    "Moscow",
    "New York",
    "A hurricane",
    "The President"
]

## Dataset

### Load

In [ ]:
train_dataloader, test_dataloader = tlmb.download_and_prepare_data(
    params['data_url'], batch_size, tokenizer, context_size
)

In [ ]:
vocab_size = len(tokenizer)
print(f"\nVocabulary size: {vocab_size}\n")

### Prepare

In [ ]:
# code here

## Model

### Build (architecture)

In [ ]:
model = tlmb.DecoderLanguageModel(vocab_size, emb_dim, num_heads, num_blocks, pad_idx)

In [ ]:
model.to(device)

### Compile

In [ ]:
# Initialize model weights using custom initialization scheme
# This is important for stable training of deep neural networks
tlmb.initialize_weights(model)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [ ]:
# Initialize the loss function (Cross Entropy) for training
# ignore_index=pad_idx ensures that padding tokens don't contribute to the loss
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)

In [ ]:
# Calculate and display the total number of trainable parameters in the model
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params / 1e6}E6\n")

### Training loop (fit)

In [ ]:
# Main training loop - iterate through specified number of epochs
for epoch in range(num_epochs):
    # Set model to training mode
    model.train()

    # Initialize tracking variables for this epoch
    total_loss = 0.0      # Accumulator for loss across all batches
    total_tokens = 0      # Counter for actual tokens processed (excluding padding)

    # Create progress bar for monitoring training progress
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")

    # Iterate through batches in the training data
    for batch_idx, (input_seq, target_seq) in enumerate(progress_bar):
        # Move input and target sequences to GPU if available
        input_seq = input_seq.to(device)
        target_seq = target_seq.to(device)

        # Clear gradients from previous batch
        optimizer.zero_grad()

        # Forward pass: get model predictions for this batch
        # output shape: (batch_size, seq_len, vocab_size)
        logits = model(input_seq)

        # Reshape logits and target tensors for loss computation
        logits = logits.reshape(-1, logits.size(-1))
        target = target_seq.reshape(-1)

        # Create mask to exclude padding tokens from loss calculation
        mask = target != pad_idx

        # Compute loss between model predictions and actual targets
        # Using masked versions to ignore padding tokens
        loss = criterion(logits[mask], target[mask])

        # Backward pass: compute gradients of loss with respect to model parameters
        loss.backward()

        # Update model parameters using calculated gradients
        optimizer.step()

        # Calculate actual loss value for this batch accounting for padding
        loss_value = loss.item() * mask.sum().item()

        # Accumulate total loss and tokens for epoch statistics
        total_loss += loss_value
        total_tokens += mask.sum().item()
        examples_processed += input_seq.size(0)

        # Update progress bar with current batch loss
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})

        # Periodic evaluation after processing specified number of examples
        if examples_processed >= eval_interval:
            # Calculate average loss over the last eval_interval examples
            avg_loss = total_loss / total_tokens
            print(f"\nAfter {examples_processed} examples, Average Loss: {avg_loss:.4f}")

            # Switch to evaluation mode
            model.eval()

            # Compute validation metrics
            average_loss, perplexity = compute_loss_and_perplexity(
                model, test_dataloader, tokenizer, criterion, device, max_sentences=1000
            )
            # Record validation
            print(f"\nValidation Average Loss: {average_loss:.4f}, Perplexity: {perplexity:.2f}")

            model.eval()

            # Generate sample texts to qualitatively assess model performance
            for context in contexts:
                # Generate text continuation for each test context
                generated_text = generate_text(
                    model=model,
                    start_string=context,
                    tokenizer=tokenizer,
                    device=device,
                    max_length=50
                )
                print(f"\nContext: {context}")
                print(f"\nGenerated text: {generated_text}\n")

            # Switch back to training mode for continued training
            model.train()

            # Reset counters for next evaluation interval
            examples_processed = 0
            total_loss = 0.0
            total_tokens = 0

    # End-of-epoch reporting
    if total_tokens > 0:
        # Calculate and display average loss for the epoch
        avg_loss = total_loss / total_tokens
        print(f"\nEpoch {epoch+1}/{num_epochs}, Average Loss: {avg_loss:.4f}")
    else:
        # Handle edge case where no tokens were processed
        print(f"\nEpoch {epoch+1}/{num_epochs} completed.")

    # Perform end-of-epoch validation
    model.eval()

    # Generate sample texts for qualitative assessment
    print("\nGenerating text based on contexts using generate_text:\n")
    for context in contexts:
        generated_text = generate_text(
            model=model,
            start_string=context,
            tokenizer=tokenizer,
            device=device,
            max_length=50
        )
        print(f"\nContext: {context}")
        print(f"\nGenerated text: {generated_text}\n")

    average_loss, perplexity = compute_loss_and_perplexity(
        model, test_dataloader, tokenizer, criterion, device, max_sentences=1000
    )
    print(f"\nValidation Average Loss: {average_loss:.4f}, Perplexity: {perplexity:.2f}")

    # Reset to training mode for next epoch
    model.train()

In [ ]:
# Save the trained model and tokenizer for later use
# This includes model architecture, weights, and tokenizer configuration
#model_name = "Decoder_LM"
model_name = "./data/trained/Decoder_LM"
save_model(model, tokenizer, model_name)

## Results

### Predictions

In [ ]:
# code here

### Evaluation (accuracy)

In [ ]:
# code here

### Show

In [ ]:
# code here

Plot wrongs